In [21]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import certifi

In [22]:
load_dotenv(override=True)

True

In [24]:
os.environ['SSL_CERT_FILE'] = certifi.where()

In [25]:
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("maneeshmukundan1379@outlook.com")  # Change to your verified sender
    to_email = To("maneeshmukundan1379@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)
    send_test_email


In [26]:
instructions1 = "You are a sales agent working for QuarkAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for QuarkAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for QuarkAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [28]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [34]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("maneeshmukundan1379@outlook.com")  # Change to your verified sender
    to_email = To("maneeshmukundan@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [35]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11319dee0>, strict_json_schema=True, is_enabled=True)

In [36]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

In [42]:
instuctions = "You are a sales manager at QuarkAI. You use the tools given to you to generate cold emails. \
You never generate emails yourself; you always use the tools. \
You should try all 3 sales agent tools before selecting the best email. \
You pick the best email and send it using the send_email tool."

sales_manager = Agent(name="Sales Manager", instructions=instuctions, tools=tools, model="gpt-4o-mini")

message = "Write a cold sales email addressed to 'Dear CEO' and end with signature as 'Garfield' If error sending email, just print the output here"

with trace("QuarkAI Cold Sales Email"):
    result = await Runner.run(sales_manager, message)

    print(f"Best sales email:\n{result.final_output}")


Best sales email:
It seems there was an error attempting to send the email. Here's the selected email:

---

**Subject:** Simplifying Your SOC2 Compliance Journey

**Dear CEO,**

I hope this message finds you well.

At QuarkAI, we recognize the challenges that come with ensuring SOC2 compliance and preparing for audits. Our AI-powered SaaS tool streamlines these processes, making compliance more efficient and less time-consuming. By automating documentation and preparation, we allow you to concentrate on scaling your business.

I would welcome the opportunity to discuss how our tailored solutions can specifically address your needs.

Looking forward to your response.

**Best regards,**  
Garfield  
[Your Position]  
QuarkAI  
[Your Contact Information]  
[Website URL]  

---

If you need further assistance, feel free to ask!
